# Phase 1: Ingestion & Vector Store

**Enterprise Agentic RAG System**  
Tech Stack: **LlamaIndex** + **Ollama (Gemma 4 Latest)** + Chroma

This notebook demonstrates a production-grade ingestion pipeline:
- Load the *AI Agents Guidebook* using `UnstructuredReader`
- Chunk with `SentenceSplitter` (parameters from `config.yaml`)
- Enrich every node with rich metadata (`page_number`, `section`, `has_code`, `has_diagram`, etc.)
- Persist to **Chroma** (idempotent)
- Build a `QueryEngine` using **Gemma 4 Latest**
- Run sample queries and inspect retrieved nodes with metadata

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

## 1. Configuration & LlamaIndex Setup

In [ ]:
from src.config import get_settings, settings
from src.logging_config import logger

print("LLM Model          :", settings.ollama.llm_model)
print("Embedding Model    :", settings.ollama.embed_model)
print("Chunk Size / Overlap:", settings.llama_index.chunk_size, "/", settings.llama_index.chunk_overlap)
print("Vector Store Dir   :", settings.llama_index.vector_store.persist_dir)
print("Collection         :", settings.llama_index.vector_store.collection_name)
print("Force Reingest     :", settings.ingestion.force_reingest)

In [ ]:
# This wires Gemma 4 Latest + nomic-embed-text into LlamaIndex global Settings
settings.configure_llama_index()

from llama_index.core import Settings as LlamaSettings
print("LlamaIndex LLM     :", type(LlamaSettings.llm).__name__, "-", LlamaSettings.llm.model)
print("LlamaIndex Embedder:", type(LlamaSettings.embed_model).__name__)

## 2. Run Ingestion Pipeline (UnstructuredReader + Chroma)

In [ ]:
from src.ingestion import GuidebookIngestionPipeline

pipeline = GuidebookIngestionPipeline()

# Run ingestion (idempotent by default)
num_nodes = pipeline.run(force=settings.ingestion.force_reingest)

print(f"\n✅ Total nodes in vector store: {num_nodes}")

## 3. Inspect a Few Ingested Nodes (Rich Metadata)

In [ ]:
import chromadb
from pathlib import Path

from src.config import _find_project_root
project_root = _find_project_root()
persist_dir = project_root / settings.llama_index.vector_store.persist_dir
client = chromadb.PersistentClient(path=str(persist_dir))
collection = client.get_collection(settings.llama_index.vector_store.collection_name)

sample = collection.get(limit=3, include=["metadatas", "documents"])

for i, (doc, meta) in enumerate(zip(sample["documents"], sample["metadatas"]), 1):
    print(f"\n=== Node {i} ===")
    print("Page        :", meta.get("page_number"))
    print("Section     :", meta.get("section"))
    print("Has Code    :", meta.get("has_code"))
    print("Has Diagram :", meta.get("has_diagram"))
    print("Source      :", meta.get("source"))
    print("Text (first 350 chars):")
    print(doc[:350] if doc else "(empty)")
    print("...")

## 4. Build QueryEngine (Gemma 4 Latest + Vector Store)

In [ ]:
from src.retrieval import get_query_engine

query_engine = get_query_engine()
print("QueryEngine ready using model:", settings.ollama.llm_model)

## 5. Test Queries + Retrieved Context Inspection

We run several representative questions from the AI Agents Guidebook domain.

In [ ]:
test_queries = [
    "What is the ReAct pattern?",
    "Explain the 5 levels of agentic systems.",
   
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}: {query}")
    print('='*70)

    response = query_engine.query(query)
    print("\nAnswer:")
    print(str(response)[:1200])
    print("...\n")

### Detailed Retrieval Inspection for One Query

Let's look at the actual nodes that were retrieved for better debugging of chunk quality and metadata.

In [ ]:
query = "What is the ReAct pattern?"
print(f"Detailed retrieval for: {query}\n")

# Get the underlying retriever to inspect source nodes
retriever = query_engine.retriever
retrieved_nodes = retriever.retrieve(query)

print(f"Retrieved {len(retrieved_nodes)} nodes:\n")

for i, node_with_score in enumerate(retrieved_nodes, 1):
    node = node_with_score.node
    meta = node.metadata
    score = node_with_score.score

    print(f"--- Node {i} | Score: {score:.4f} ---")
    print(f"Page       : {meta.get('page_number')}")
    print(f"Section    : {meta.get('section')}")
    print(f"Has Code   : {meta.get('has_code')} | Has Diagram: {meta.get('has_diagram')}")
    print(f"Text preview:\n{node.get_content()[:450]}...\n")

## 6. Summary & Next Steps

- ✅ PDF loaded with `UnstructuredReader` (good layout preservation)
- ✅ Nodes chunked using settings from `config.yaml`
- ✅ Every node carries rich metadata (`page_number`, `section`, `has_code`, `has_diagram`)
- ✅ Persisted to Chroma at `data/processed/chroma_db/`
- ✅ QueryEngine built with **Gemma 4 Latest**

**Next phases ideas:**
- Add better heading extraction / hierarchical nodes
- Implement hybrid search (BM25 + vector)
- Add reranking (Cohere or local cross-encoder via Ollama)
- Build a proper `IngestionPipeline` class with more transformations
- Start evaluation with Ragas on these queries

You can now use `get_query_engine()` from anywhere in the project.